Prepare the Dataset

In [1]:
"""
Prepare the Shakespeare dataset for character-level language modeling.
So instead of encoding with GPT-2 BPE tokens, we just map characters to ints.
Will save train.bin, val.bin containing the ids, and meta.pkl containing the
encoder and decoder and some other related info.
"""
import os
import pickle
import requests
import numpy as np

# download the tiny shakespeare dataset
directory_path = os.path.join(os.path.abspath(''),"data\shakespeare_char")
input_file_path = os.path.join(directory_path, 'input.txt')

print(directory_path)

if not os.path.exists(directory_path):
    os.makedirs(directory_path)
if not os.path.exists(input_file_path):
    data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    with open(input_file_path, 'w') as f:
        f.write(requests.get(data_url).text)

with open(input_file_path, 'r') as f:
    data = f.read()
print(f"length of dataset in characters: {len(data):,}")

# get all the unique characters that occur in this text
chars = sorted(list(set(data)))
vocab_size = len(chars)
print("all the unique characters:", ''.join(chars))
print(f"vocab size: {vocab_size:,}")

# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
def encode(s):
    return [stoi[c] for c in s] # encoder: take a string, output a list of integers
def decode(l):
    return ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# create the train and test splits
n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

# encode both to integers
train_ids = encode(train_data)
val_ids = encode(val_data)
print(f"train has {len(train_ids):,} tokens")
print(f"val has {len(val_ids):,} tokens")

# export to bin files
train_ids = np.array(train_ids, dtype=np.uint16)
val_ids = np.array(val_ids, dtype=np.uint16)
train_ids.tofile(os.path.join(directory_path, 'train.bin'))
val_ids.tofile(os.path.join(directory_path, 'val.bin'))

# save the meta information as well, to help us encode/decode later
meta = {
    'vocab_size': vocab_size,
    'itos': itos,
    'stoi': stoi,
}
with open(os.path.join(directory_path, 'meta.pkl'), 'wb') as f:
    pickle.dump(meta, f)

# length of dataset in characters:  1115394
# all the unique characters:
#  !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
# vocab size: 65
# train has 1003854 tokens
# val has 111540 tokens


c:\Users\lunat\Documents\GitHub\foundations_NLP\final_assignment\data\shakespeare_char
length of dataset in characters: 1,115,394
all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens


Baseline Configuration

In [ ]:
#Baseline configuration
out_dir = 'out-shakespeare-baseline'
eval_interval = 250
eval_iters = 200
log_interval = 10
always_save_checkpoint = False
wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'baseline'
dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2
learning_rate = 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100
weight_decay = 1e-1
device = 'cuda' # change to 'cuda' if you have a GPU
compile = False # set True only on Linux with GPU

#for testing
num_samples=3
max_new_tokens=500

In [3]:
"""
This training script can be run both on a single gpu in debug mode,
and also in a larger training run with distributed data parallel (ddp).

To run on a single GPU, example:
$ python train.py --batch_size=32 --compile=False

To run with DDP on 4 gpus on 1 node, example:
$ torchrun --standalone --nproc_per_node=4 train.py

To run with DDP on 4 gpus across 2 nodes, example:
- Run on the first (master) node with example IP 123.456.123.456:
$ torchrun --nproc_per_node=8 --nnodes=2 --node_rank=0 --master_addr=123.456.123.456 --master_port=1234 train.py
- Run on the worker node:
$ torchrun --nproc_per_node=8 --nnodes=2 --node_rank=1 --master_addr=123.456.123.456 --master_port=1234 train.py
(If your cluster does not have Infiniband interconnect prepend NCCL_IB_DISABLE=1)
"""

import os
import time
import math
import pickle
from contextlib import nullcontext

import numpy as np
import torch
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group

from model import GPTConfig, GPT

# -----------------------------------------------------------------------------
# default config values designed to train a gpt2 (124M) on OpenWebText
# I/O
eval_only = False # if True, script exits right after the first eval
init_from = 'scratch' # 'scratch' or 'resume' or 'gpt2*'
# model
bias = False # do we use bias inside LayerNorm and Linear layers?
# adamw optimizer
beta1 = 0.9
grad_clip = 1.0 # clip gradients at this value, or disable if == 0.0
# learning rate decay settings
decay_lr = True # whether to decay the learning rate
lr_decay_iters = 600000 # should be ~= max_iters per Chinchilla
# DDP settings
backend = 'nccl' # 'nccl', 'gloo', etc.
# system
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler

def run():
    global gradient_accumulation_steps, device
    # -----------------------------------------------------------------------------
    config_keys = [k for k,v in globals().items() if not k.startswith('_') and isinstance(v, (int, float, bool, str))]
    config = {k: globals()[k] for k in config_keys} # will be useful for logging
    # -----------------------------------------------------------------------------

    # various inits, derived attributes, I/O setup
    ddp = int(os.environ.get('RANK', -1)) != -1 # is this a ddp run?
    if ddp:
        init_process_group(backend=backend)
        ddp_rank = int(os.environ['RANK'])
        ddp_local_rank = int(os.environ['LOCAL_RANK'])
        ddp_world_size = int(os.environ['WORLD_SIZE'])
        device = f'cuda:{ddp_local_rank}'
        torch.cuda.set_device(device)
        master_process = ddp_rank == 0 # this process will do logging, checkpointing etc.
        seed_offset = ddp_rank # each process gets a different seed
        # world_size number of processes will be training simultaneously, so we can scale
        # down the desired gradient accumulation iterations per process proportionally
        assert gradient_accumulation_steps % ddp_world_size == 0
        gradient_accumulation_steps //= ddp_world_size
    else:
        # if not ddp, we are running on a single gpu, and one process
        master_process = True
        seed_offset = 0
        ddp_world_size = 1
    tokens_per_iter = gradient_accumulation_steps * ddp_world_size * batch_size * block_size
    print(f"tokens per iteration will be: {tokens_per_iter:,}")

    if master_process:
        os.makedirs(out_dir, exist_ok=True)
    torch.manual_seed(1337 + seed_offset)
    torch.backends.cuda.matmul.allow_tf32 = True # allow tf32 on matmul
    torch.backends.cudnn.allow_tf32 = True # allow tf32 on cudnn
    device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
    # note: float16 data type will automatically use a GradScaler
    ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
    ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

    # poor man's data loader
    data_dir = os.path.join('data', dataset)
    def get_batch(split):
        # We recreate np.memmap every batch to avoid a memory leak, as per
        # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
        if split == 'train':
            data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
        else:
            data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint16, mode='r')
        ix = torch.randint(len(data) - block_size, (batch_size,))
        x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
        if device_type == 'cuda':
            # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
            x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
        else:
            x, y = x.to(device), y.to(device)
        return x, y

    # init these up here, can override if init_from='resume' (i.e. from a checkpoint)
    iter_num = 0
    best_val_loss = 1e9

    # attempt to derive vocab_size from the dataset
    meta_path = os.path.join(data_dir, 'meta.pkl')
    meta_vocab_size = None
    if os.path.exists(meta_path):
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        meta_vocab_size = meta['vocab_size']
        print(f"found vocab_size = {meta_vocab_size} (inside {meta_path})")

    # model init
    model_args = dict(n_layer=n_layer, n_head=n_head, n_embd=n_embd, block_size=block_size,
                    bias=bias, vocab_size=None, dropout=dropout) # start with model_args from command line
    if init_from == 'scratch':
        # init a new model from scratch
        print("Initializing a new model from scratch")
        # determine the vocab size we'll use for from-scratch training
        if meta_vocab_size is None:
            print("defaulting to vocab_size of GPT-2 to 50304 (50257 rounded up for efficiency)")
        model_args['vocab_size'] = meta_vocab_size if meta_vocab_size is not None else 50304
        gptconf = GPTConfig(**model_args)
        model = GPT(gptconf)
    elif init_from == 'resume':
        print(f"Resuming training from {out_dir}")
        # resume training from a checkpoint.
        ckpt_path = os.path.join(out_dir, 'ckpt.pt')
        checkpoint = torch.load(ckpt_path, map_location=device)
        checkpoint_model_args = checkpoint['model_args']
        # force these config attributes to be equal otherwise we can't even resume training
        # the rest of the attributes (e.g. dropout) can stay as desired from command line
        for k in ['n_layer', 'n_head', 'n_embd', 'block_size', 'bias', 'vocab_size']:
            model_args[k] = checkpoint_model_args[k]
        # create the model
        gptconf = GPTConfig(**model_args)
        model = GPT(gptconf)
        state_dict = checkpoint['model']
        # fix the keys of the state dictionary :(
        # honestly no idea how checkpoints sometimes get this prefix, have to debug more
        unwanted_prefix = '_orig_mod.'
        for k,v in list(state_dict.items()):
            if k.startswith(unwanted_prefix):
                state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
        model.load_state_dict(state_dict)
        iter_num = checkpoint['iter_num']
        best_val_loss = checkpoint['best_val_loss']
    elif init_from.startswith('gpt2'):
        print(f"Initializing from OpenAI GPT-2 weights: {init_from}")
        # initialize from OpenAI GPT-2 weights
        override_args = dict(dropout=dropout)
        model = GPT.from_pretrained(init_from, override_args)
        # read off the created config params, so we can store them into checkpoint correctly
        for k in ['n_layer', 'n_head', 'n_embd', 'block_size', 'bias', 'vocab_size']:
            model_args[k] = getattr(model.config, k)
    # crop down the model block size if desired, using model surgery
    if block_size < model.config.block_size:
        model.crop_block_size(block_size)
        model_args['block_size'] = block_size # so that the checkpoint will have the right value
    model.to(device)

    # initialize a GradScaler. If enabled=False scaler is a no-op
    scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

    # optimizer
    optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)
    if init_from == 'resume':
        optimizer.load_state_dict(checkpoint['optimizer'])
    checkpoint = None # free up memory

    # compile the model
    if compile:
        print("compiling the model... (takes a ~minute)")
        unoptimized_model = model
        model = torch.compile(model) # requires PyTorch 2.0

    # wrap model into DDP container
    if ddp:
        model = DDP(model, device_ids=[ddp_local_rank])

    # helps estimate an arbitrarily accurate loss over either split using many batches
    @torch.no_grad()
    def estimate_loss():
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
        model.train()
        return out

    # learning rate decay scheduler (cosine with warmup)
    def get_lr(it):
        # 1) linear warmup for warmup_iters steps
        if it < warmup_iters:
            return learning_rate * (it + 1) / (warmup_iters + 1)
        # 2) if it > lr_decay_iters, return min learning rate
        if it > lr_decay_iters:
            return min_lr
        # 3) in between, use cosine decay down to min learning rate
        decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
        assert 0 <= decay_ratio <= 1
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff ranges 0..1
        return min_lr + coeff * (learning_rate - min_lr)

    # logging
    if wandb_log and master_process:
        import wandb
        wandb.init(project=wandb_project, name=wandb_run_name, config=config)

    # training loop
    X, Y = get_batch('train') # fetch the very first batch
    t0 = time.time()
    local_iter_num = 0 # number of iterations in the lifetime of this process
    raw_model = model.module if ddp else model # unwrap DDP container if needed
    running_mfu = -1.0

    history = {
        "steps": [],
        "train_loss": [],
        "val_loss": []
    }
    
    while True:

        # determine and set the learning rate for this iteration
        lr = get_lr(iter_num) if decay_lr else learning_rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # evaluate the loss on train/val sets and write checkpoints
        if iter_num % eval_interval == 0 and master_process:
            losses = estimate_loss()
            history["steps"].append(iter_num)
            history["train_loss"].append(float(losses["train"]))
            history["val_loss"].append(float(losses["val"]))

            print(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
            if wandb_log:
                wandb.log({
                    "iter": iter_num,
                    "train/loss": losses['train'],
                    "val/loss": losses['val'],
                    "lr": lr,
                    "mfu": running_mfu*100, # convert to percentage
                })
            if losses['val'] < best_val_loss or always_save_checkpoint:
                best_val_loss = losses['val']
                if iter_num > 0:
                    checkpoint = {
                        'model': raw_model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'model_args': model_args,
                        'iter_num': iter_num,
                        'best_val_loss': best_val_loss,
                        'config': config,
                    }
                    print(f"saving checkpoint to {out_dir}")
                    torch.save(checkpoint, os.path.join(out_dir, 'ckpt.pt'))
        if iter_num == 0 and eval_only:
            break

        # forward backward update, with optional gradient accumulation to simulate larger batch size
        # and using the GradScaler if data type is float16
        for micro_step in range(gradient_accumulation_steps):
            if ddp:
                # in DDP training we only need to sync gradients at the last micro step.
                # the official way to do this is with model.no_sync() context manager, but
                # I really dislike that this bloats the code and forces us to repeat code
                # looking at the source of that context manager, it just toggles this variable
                model.require_backward_grad_sync = (micro_step == gradient_accumulation_steps - 1)
            with ctx:
                logits, loss = model(X, Y)
                loss = loss / gradient_accumulation_steps # scale the loss to account for gradient accumulation
            # immediately async prefetch next batch while model is doing the forward pass on the GPU
            X, Y = get_batch('train')
            # backward pass, with gradient scaling if training in fp16
            scaler.scale(loss).backward()
        # clip the gradient
        if grad_clip != 0.0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        # step the optimizer and scaler if training in fp16
        scaler.step(optimizer)
        scaler.update()
        # flush the gradients as soon as we can, no need for this memory anymore
        optimizer.zero_grad(set_to_none=True)

        # timing and logging
        t1 = time.time()
        dt = t1 - t0
        t0 = t1
        if iter_num % log_interval == 0 and master_process:
            # get loss as float. note: this is a CPU-GPU sync point
            # scale up to undo the division above, approximating the true total loss (exact would have been a sum)
            lossf = loss.item() * gradient_accumulation_steps
            if local_iter_num >= 5: # let the training loop settle a bit
                mfu = raw_model.estimate_mfu(batch_size * gradient_accumulation_steps, dt)
                running_mfu = mfu if running_mfu == -1.0 else 0.9*running_mfu + 0.1*mfu
            print(f"iter {iter_num}: loss {lossf:.4f}, time {dt*1000:.2f}ms, mfu {running_mfu*100:.2f}%")
        iter_num += 1
        local_iter_num += 1

        # termination conditions
        if iter_num > max_iters:
            break

    if ddp:
        destroy_process_group()

    return history



In [4]:
import os
import pickle
from contextlib import nullcontext
import torch
import tiktoken

from model import GPTConfig, GPT


def generate_and_save_samples(
    out_dir,
    output_file=None,
    start="\n",
    num_samples=3,
    max_new_tokens=500,
    temperature=0.8,
    top_k=200,
    seed=1337,
    device="cpu",
    dtype=None,
    compile_model=False,
    print_to_notebook=True
):
    """
    Load a trained nanoGPT checkpoint from out_dir, generate text samples,
    and save them to a text file.

    Returns
    -------
    samples : list of str
        Generated text samples.
    """

    if dtype is None:
        if device.startswith("cuda"):
            if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
                dtype = "bfloat16"
            else:
                dtype = "float16"
        else:
            dtype = "float32"

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    device_type = "cuda" if "cuda" in device else "cpu"
    ptdtype = {
        "float32": torch.float32,
        "bfloat16": torch.bfloat16,
        "float16": torch.float16
    }[dtype]

    ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

    # -------- load checkpoint --------
    ckpt_path = os.path.join(out_dir, "ckpt.pt")
    checkpoint = torch.load(ckpt_path, map_location=device)

    gptconf = GPTConfig(**checkpoint["model_args"])
    model = GPT(gptconf)

    state_dict = checkpoint["model"]
    unwanted_prefix = "_orig_mod."
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model.load_state_dict(state_dict)
    model.eval()
    model.to(device)

    if compile_model:
        model = torch.compile(model)

    # -------- tokenizer / decoder --------
    load_meta = False
    if "config" in checkpoint and "dataset" in checkpoint["config"]:
        meta_path = os.path.join("data", checkpoint["config"]["dataset"], "meta.pkl")
        load_meta = os.path.exists(meta_path)
    else:
        meta_path = None

    if load_meta:
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)
        stoi, itos = meta["stoi"], meta["itos"]
        encode = lambda s: [stoi[c] for c in s]
        decode = lambda l: "".join([itos[i] for i in l])
    else:
        enc = tiktoken.get_encoding("gpt2")
        encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
        decode = lambda l: enc.decode(l)

    # -------- prompt --------
    if start.startswith("FILE:"):
        with open(start[5:], "r", encoding="utf-8") as f:
            start = f.read()

    start_ids = encode(start)
    x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]

    # -------- generate --------
    samples = []
    with torch.no_grad():
        with ctx:
            for k in range(num_samples):
                y = model.generate(
                    x,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    top_k=top_k
                )
                text = decode(y[0].tolist())
                samples.append(text)

                if print_to_notebook:
                    print(f"=== Sample {k+1} ===")
                    print(text)
                    print("-" * 60)

    # -------- save --------
    if output_file is None:
        output_file = os.path.join(out_dir, "generated_samples.txt")

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"Checkpoint directory: {out_dir}\n")
        f.write(f"Prompt: {repr(start)}\n")
        f.write(f"num_samples={num_samples}, max_new_tokens={max_new_tokens}, "
                f"temperature={temperature}, top_k={top_k}, device={device}\n")
        f.write("=" * 80 + "\n\n")

        for i, text in enumerate(samples, start=1):
            f.write(f"=== Sample {i} ===\n")
            f.write(text)
            f.write("\n\n" + "-" * 80 + "\n\n")

    print(f"Saved generated samples to: {output_file}")
    return samples

def generate_for_run(run_name):
    global device, num_samples, max_new_tokens
    return generate_and_save_samples(
        out_dir=run_name,
        output_file=f"{run_name}_generated_samples.txt",
        device=device,
        num_samples=num_samples,
        max_new_tokens=max_new_tokens
    )

Baseline setup

In [6]:
all_runs = {}
all_samples = {}

run_name = "baseline" 
wandb_run_name = run_name
out_dir = run_name
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

tokens per iteration will be: 16,384
found vocab_size = 65 (inside data\shakespeare_char\meta.pkl)
Initializing a new model from scratch
number of parameters: 10.65M
num decayed parameter tensors: 26, with 10,740,096 parameters
num non-decayed parameter tensors: 13, with 4,992 parameters
using fused AdamW: True


C:\Users\lunat\AppData\Local\Temp\ipykernel_20348\2101442384.py:172: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


step 0: train loss 4.2874, val loss 4.2823
iter 0: loss 4.2670, time 4298.68ms, mfu -100.00%
iter 10: loss 3.1456, time 30.04ms, mfu 12.41%
iter 20: loss 2.7262, time 33.03ms, mfu 12.29%
iter 30: loss 2.6158, time 31.53ms, mfu 12.25%
iter 40: loss 2.5681, time 31.20ms, mfu 12.22%
iter 50: loss 2.5237, time 31.12ms, mfu 12.19%
iter 60: loss 2.5118, time 30.39ms, mfu 12.20%
iter 70: loss 2.4919, time 29.51ms, mfu 12.24%
iter 80: loss 2.4971, time 40.88ms, mfu 11.93%
iter 90: loss 2.4629, time 30.71ms, mfu 11.95%
iter 100: loss 2.4936, time 17.68ms, mfu 12.86%
iter 110: loss 2.4521, time 30.31ms, mfu 12.81%
iter 120: loss 2.4279, time 31.02ms, mfu 12.73%
iter 130: loss 2.4143, time 31.21ms, mfu 12.65%
iter 140: loss 2.4037, time 31.04ms, mfu 12.58%
iter 150: loss 2.4159, time 17.83ms, mfu 13.42%
iter 160: loss 2.3707, time 30.58ms, mfu 13.29%
iter 170: loss 2.3562, time 18.05ms, mfu 14.03%
iter 180: loss 2.3124, time 29.57ms, mfu 13.89%
iter 190: loss 2.2434, time 18.24ms, mfu 14.54%
step

TypeError: 'tuple' object cannot be interpreted as an integer

Learning Rate

In [ ]:
learning_rate_old = learning_rate

run_name = "lr-1e-4"
wandb_run_name = run_name
out_dir = run_name
learning_rate = 1e-4 
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "lr-5e-4"
wandb_run_name = run_name
out_dir = run_name
learning_rate = 5e-4 
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "lr-5e-3"
wandb_run_name = run_name
out_dir = run_name
learning_rate = 5e-3 
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "lr-1e-2"
wandb_run_name = run_name
out_dir = run_name
learning_rate = 1e-2 
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)
learning_rate = learning_rate_old

Depth

In [ ]:
n_layer_old = n_layer
run_name = "depth-2"
wandb_run_name = run_name
out_dir = run_name
n_layer = 2 
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "depth-4"
wandb_run_name = run_name
out_dir = run_name
n_layer = 4 
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "depth-8"
wandb_run_name = run_name
out_dir = run_name
n_layer = 8 
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)
n_layer = n_layer_old

Embd & Heads

In [ ]:
n_embd_base = n_embd
n_head_base = n_head

run_name = "128-4heads"
wandb_run_name = run_name
out_dir = run_name
n_embd = 128
n_head = 4
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "256-4heads"
wandb_run_name = run_name
out_dir = run_name
n_embd = 256
n_head = 4
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

n_embd = n_embd_base
n_head = n_head_base

Blocksize

In [ ]:
block_size_base = block_size

run_name = "block-64"
wandb_run_name = run_name
out_dir = run_name
block_size = 64
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "block-128"
wandb_run_name = run_name
out_dir = run_name
block_size = 128
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

block_size = block_size_base

Dropout

In [ ]:
dropout_base = dropout

run_name = "dropout-0-0"
wandb_run_name = run_name
out_dir = run_name
dropout = 0.0
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "dropout-0-1"
wandb_run_name = run_name
out_dir = run_name
dropout = 0.1
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "dropout-0-4"
wandb_run_name = run_name
out_dir = run_name
dropout = 0.4
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

dropout = dropout_base

Iterations

In [ ]:
max_iters_base = max_iters

run_name = "td1000"
wandb_run_name = run_name
out_dir = run_name
max_iters = 1000
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "td2500"
wandb_run_name = run_name
out_dir = run_name
max_iters = 2500
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)

run_name = "td10000"
wandb_run_name = run_name
out_dir = run_name
max_iters = 10000
all_runs[run_name] = run()
all_samples[run_name] = generate_for_run(run_name=run_name)



Save generated data for other users or for later analytics

In [ ]:
import pickle

with open("all_runs.pkl", "wb") as f:
    pickle.dump(all_runs, f)

with open("all_rsample.pkl", "wb") as f:
    pickle.dump(all_samples, f)

Load the Data 

In [ ]:
import pickle

with open("all_runs.pkl", "rb") as f:
    all_runs = pickle.load(f)

with open("all_rsample.pkl", "rb") as f:
    all_samples = pickle.load(f)

Setup function for ploting different states 

In [ ]:
import matplotlib.pyplot as plt

def plot_runs(
    all_runs,
    show_train=True,
    show_val=True,
    selected_runs=None,
    figsize=(10, 6),
    save_plots=False,
    train_filename="training_loss_vs_step.png",
    val_filename="validation_loss_vs_step.png"
):
    """
    Plot train/val loss curves from stored run histories.

    Parameters
    ----------
    all_runs : dict
        Example:
        {
            "baseline": {"steps": [...], "train_loss": [...], "val_loss": [...]},
            "lr_1e-4": {"steps": [...], "train_loss": [...], "val_loss": [...]}
        }

    show_train : bool
        Whether to show the training loss plot.

    show_val : bool
        Whether to show the validation loss plot.

    selected_runs : list or None
        If given, only these run names are plotted.

    figsize : tuple
        Figure size for the plots.

    save_plots : bool
        Whether to save the figures as PNG files.

    train_filename : str
        Output filename for training plot.

    val_filename : str
        Output filename for validation plot.
    """

    if selected_runs is None:
        selected_runs = list(all_runs.keys())

    if show_train:
        plt.figure(figsize=figsize)
        for run_name in selected_runs:
            if run_name not in all_runs:
                print(f"Warning: '{run_name}' not found in all_runs")
                continue
            history = all_runs[run_name]
            plt.plot(history["steps"], history["train_loss"], label=run_name)

        plt.xlabel("Step")
        plt.ylabel("Training Loss")
        plt.title("Training Loss vs. Step")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        if save_plots:
            plt.savefig(train_filename, dpi=200)

        plt.show()

    if show_val:
        plt.figure(figsize=figsize)
        for run_name in selected_runs:
            if run_name not in all_runs:
                print(f"Warning: '{run_name}' not found in all_runs")
                continue
            history = all_runs[run_name]
            plt.plot(history["steps"], history["val_loss"], label=run_name)

        plt.xlabel("Step")
        plt.ylabel("Validation Loss")
        plt.title("Validation Loss vs. Step")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        if save_plots:
            plt.savefig(val_filename, dpi=200)

        plt.show()

Show Train Loss

In [ ]:
plot_runs(all_runs, show_train=True, show_val=False)

Show Validation Loss

In [ ]:
plot_runs(all_runs, show_train=False, show_val=True)

How To select specific runs

In [ ]:
plot_runs(all_runs, selected_runs=["baseline", "lr-1e-4"])